## Multilevel bambi

# GENDER | COMPONENT

In [ ]:
import pandas as pd
import bambi as bmb
from anthropmass.anthro_module import *

In [ ]:
train=pd.read_csv('../data/processed/ANSURIImalefemaleastext.csv')

Setting priors for the  model

In [ ]:
def setpriors_com(measurement, data):

    mean = data[measurement].mean()
    sd = data[measurement].std()

    mean_weight = data['weightkg'].mean()   
    mean_stature = data['stature'].mean()   

    priors = {
        "Intercept": bmb.Prior("Normal", mu=mean, sigma=sd*3),
        "Gender": bmb.Prior("Normal", mu=0, sigma=1),
        "weightkg": bmb.Prior("Normal", mu=mean/mean_weight, sigma=1),
        "stature": bmb.Prior("Normal", mu=mean/mean_stature, sigma=1),
        "Gender|Component": bmb.Prior("Normal", mu=0, sigma=bmb.Prior("Exponential", lam=1)),
        "sigma": bmb.Prior("Exponential", lam=1),
    }
    return priors
    

Multilevel model with Gender|Component

In [ ]:
def component_model(measurement, data):
    model = bmb.Model(
    formula=f"{measurement} ~ 1 + Gender + weightkg + stature + (0 + Gender|Component)",
    data=data,
    priors=setpriors_com(measurement, data),
    noncentered=True
    )

    fitted_model = model.fit(draws=2000, tune=2000,target_accept=0.9, chains=4)

    return fitted_model, model


Making the model and fitting it 

In [ ]:
fitted_model, model = component_model("neckcircumference", train)